<a href="https://colab.research.google.com/github/sairamsrujan/celebal-excellence-internship/blob/main/Week8_RSaiRamSrujanKumar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-Agent Smart Assistant

A small agent that reads a user query, works out the intent, and routes it to the right
tool — a calculator, a keyword extractor, or a plain text reply — and always answers in a
fixed JSON shape: `{"type": ..., "result": ...}`.

**Required:** agent logic, conditional routing, tool integration, error handling.
**Bonus attempted:** better routing, logging, an extra tool, plus a retry loop, a
trajectory log and simple metrics.

### How I thought about it (linking back to the theory questions)
The `agent()` function is really a tiny *stateful directed graph*. A query enters one
router node, gets pushed along one edge to a tool node, and the answer comes back in a
consistent schema. I deliberately built in the pieces the quiz asks about so the code and
the written answers line up:

| Quiz idea | Where it shows up in the code |
|---|---|
| Conditional routing (Q3) | keyword checks inside `agent()` |
| Cycles / retry loops (Q4) | `run_with_retry()` |
| JSON-schema style I/O (Q6) | every reply goes through `_make_response()` |
| Trajectory evaluation (Q9) | `TRAJECTORY` records the path each query took |
| Completion rate + cost (Q10) | the `METRICS` counter |


In [1]:
import ast
import json
import logging
import operator
import re
from collections import Counter

# Logging so I can watch which node handled a query while debugging.
# Kept at WARNING by default so the test output stays clean JSON;
# flip to logging.INFO to see the routing decisions live.
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("agent")

# Every response must be one of these types -> keeps the output schema honest.
RESPONSE_TYPES = {"calculation", "keywords", "stats", "general", "error"}

# Run-time metrics (Q10). "cost" here = number of tool calls, the cheap proxy we control.
METRICS = {"total": 0, "completed": 0, "tool_calls": 0}

# Trajectory (Q9): remember the path each query took, not just the final answer.
TRAJECTORY = []

## Tool 1 — Calculator

The starter used `eval()` straight on the input, which will run *any* Python, not just
math (e.g. `__import__("os").system(...)`). I kept the same `calculator(expression)`
signature but parse the expression with `ast` and only allow arithmetic operators — same
result for real math, but it can't be tricked into running arbitrary code.

In [2]:
# Only these operators are allowed through the calculator.
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    """Walk the parsed expression and compute it, rejecting anything that
    isn't a plain number or an allowed arithmetic operation."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")


def calculator(expression: str) -> str:
    """Evaluate a math expression safely (no raw eval)."""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        # show 25 instead of 25.0 when the answer is a whole number
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return str(result)
    except ZeroDivisionError:
        return "Error: division by zero"
    except Exception:
        return "Error in calculation"

## Tool 2 — Keyword Extractor

The starter just kept words longer than 4 characters, which leaks filler words like
"about" / "would". I added a small stop-word list and rank by frequency so the keywords
are a bit more meaningful. Still returns a `list`.

In [3]:
# Common filler words we don't want surfacing as "keywords".
_STOPWORDS = {
    "about", "above", "after", "again", "their", "there", "these", "those",
    "which", "while", "would", "could", "should", "being", "between", "into",
    "from", "that", "this", "with", "have", "will", "your", "what", "when",
}


def extract_keywords(text: str) -> list:
    """Return up to 5 keywords, most frequent first."""
    try:
        words = re.findall(r"[a-zA-Z]+", text.lower())
        candidates = [w for w in words if len(w) > 4 and w not in _STOPWORDS]
        return [w for w, _ in Counter(candidates).most_common(5)]
    except Exception:
        return []

## Bonus Tool — Text Stats

A small extra tool (the brief suggested "add more tools"). Given some text it reports word
count, character count and the longest word. Triggered by queries containing "count" or
"stats".

In [4]:
def text_stats(text: str) -> dict:
    """Quick word / character stats for a chunk of text."""
    words = re.findall(r"[a-zA-Z']+", text)
    return {
        "words": len(words),
        "characters": len(text),
        "longest_word": max(words, key=len) if words else None,
    }

## Routing helpers

Two helpers before the agent itself: one pulls the arithmetic out of a sentence, and one
retries a tool call if it throws (Q4 — cycles / retry loops). Every tool call also goes
through the retry wrapper, which is a handy single place to count tool calls for the cost
metric.

In [5]:
def parse_expression(query: str) -> str:
    """Pull the arithmetic part out of a sentence like 'Calculate 20 + 5'."""
    # keep digits, operators, brackets, decimal points and spaces; drop the rest
    expr = re.sub(r"[^0-9+\-*/%().\s]", " ", query)
    return expr.strip()


def run_with_retry(tool, arg, attempts: int = 2):
    """Call a tool, retrying once if it raises. A transient failure (say a flaky
    network tool) shouldn't kill the whole request. Also the one spot where we
    bump the tool-call counter, so 'cost' stays consistent."""
    last_error = None
    for i in range(1, attempts + 1):
        try:
            METRICS["tool_calls"] += 1
            return tool(arg)
        except Exception as e:
            last_error = e
            log.warning("tool %s failed on attempt %d: %s", tool.__name__, i, e)
    raise last_error

## The agent

Conditional routing exactly as the brief describes — check the lowercased query for
`calculate`, then `keywords`, then my bonus `count`/`stats` route, else fall back to a
general reply. `agent()` never lets an exception escape: bad input comes back with an
explicit `"error"` type instead of crashing.

In [6]:
def _make_response(rtype: str, result) -> dict:
    """Build and sanity-check the standard response envelope."""
    assert rtype in RESPONSE_TYPES, f"unknown response type: {rtype}"
    return {"type": rtype, "result": result}


def agent(query: str) -> dict:
    """Route a query to the right tool and return a JSON-serialisable dict:
        {"type": <calculation|keywords|stats|general|error>, "result": ...}
    """
    METRICS["total"] += 1

    # guard non-string / empty input -> explicit error label
    if not isinstance(query, str) or not query.strip():
        TRAJECTORY.append({"query": query, "route": "error", "type": "error"})
        return _make_response("error", "Empty or invalid query")

    q = query.lower()
    route = "general"
    try:
        if "calculate" in q:
            route = "calculation"
            expr = parse_expression(query)
            if not expr:
                response = _make_response("error", "No expression found to calculate")
            else:
                answer = run_with_retry(calculator, expr)
                # calculator returns an "Error..." string when it can't parse
                if answer.startswith("Error"):
                    response = _make_response("error", answer)
                else:
                    response = _make_response("calculation", answer)

        elif "keywords" in q:
            route = "keywords"
            # strip the instruction part ("extract keywords from ..."), keep the text
            text = re.sub(r".*keywords\s*(from|of|in|:)?", "", query, flags=re.IGNORECASE).strip()
            response = _make_response("keywords", run_with_retry(extract_keywords, text or query))

        elif re.search(r"\bcount\b", q) or re.search(r"\bstats\b", q):
            route = "stats"
            text = re.sub(r".*(count|stats)\s*(of|for|in|:)?", "", query, flags=re.IGNORECASE).strip()
            response = _make_response("stats", run_with_retry(text_stats, text or query))

        else:
            route = "general"
            response = _make_response(
                "general",
                f"I don't have a dedicated tool for that yet. You asked: {query!r}",
            )

    except Exception as e:
        # last line of defence: nothing should escape agent()
        log.exception("agent failed on query %r", query)
        response = _make_response("error", f"Unexpected failure: {e}")

    if response["type"] != "error":
        METRICS["completed"] += 1

    log.info("router -> %s", route)
    TRAJECTORY.append({"query": query, "route": route, "type": response["type"]})
    return response

## Expected output format

```json
{"type": "calculation | keywords | stats | general | error", "result": ...}
```

## Automated validation

A mix of every route plus a few deliberately broken inputs (divide-by-zero, malformed
math, empty string) to prove the agent degrades gracefully instead of crashing. Printing
with `json.dumps` guarantees each response is a valid JSON object with both keys.

In [7]:
# reset counters so re-running this cell gives clean numbers
TRAJECTORY.clear()
METRICS.update(total=0, completed=0, tool_calls=0)

test_queries = [
    "Calculate 20 + 5",
    "Calculate (10 * 3) - 4 / 2",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "Count words in this short sentence",
    "What is machine learning?",
    "Calculate 10 / 0",          # divide-by-zero -> error, not a crash
    "calculate 5 +* 2",          # malformed expression -> error label
    "",                           # empty input -> error label
]

for q in test_queries:
    print("Query   :", q if q else "<empty>")
    print("Response:", json.dumps(agent(q)))
    print("-" * 64)

Query   : Calculate 20 + 5
Response: {"type": "calculation", "result": "25"}
----------------------------------------------------------------
Query   : Calculate (10 * 3) - 4 / 2
Response: {"type": "calculation", "result": "28"}
----------------------------------------------------------------
Query   : Extract keywords from Artificial Intelligence is transforming industries
Response: {"type": "keywords", "result": ["artificial", "intelligence", "transforming", "industries"]}
----------------------------------------------------------------
Query   : Count words in this short sentence
Response: {"type": "stats", "result": {"words": 5, "characters": 28, "longest_word": "sentence"}}
----------------------------------------------------------------
Query   : What is machine learning?
Response: {"type": "general", "result": "I don't have a dedicated tool for that yet. You asked: 'What is machine learning?'"}
----------------------------------------------------------------
Query   : Calculate 

## Trajectory & metrics

The two evaluation ideas from the quiz, made concrete. The trajectory shows the decision
path for every query (useful for debugging a wrong answer). The metrics give a completion
rate and a simple cost proxy (tool calls).

In [8]:
print("Trajectory (path taken per query):")
print(json.dumps(TRAJECTORY, indent=2))

rate = (METRICS["completed"] / METRICS["total"] * 100) if METRICS["total"] else 0
print("\nMetrics:")
print(json.dumps({
    "total_queries": METRICS["total"],
    "completed": METRICS["completed"],
    "completion_rate": f"{rate:.0f}%",
    "tool_calls_cost": METRICS["tool_calls"],
}, indent=2))

Trajectory (path taken per query):
[
  {
    "query": "Calculate 20 + 5",
    "route": "calculation",
    "type": "calculation"
  },
  {
    "query": "Calculate (10 * 3) - 4 / 2",
    "route": "calculation",
    "type": "calculation"
  },
  {
    "query": "Extract keywords from Artificial Intelligence is transforming industries",
    "route": "keywords",
    "type": "keywords"
  },
  {
    "query": "Count words in this short sentence",
    "route": "stats",
    "type": "stats"
  },
  {
    "query": "What is machine learning?",
    "route": "general",
    "type": "general"
  },
  {
    "query": "Calculate 10 / 0",
    "route": "calculation",
    "type": "error"
  },
  {
    "query": "calculate 5 +* 2",
    "route": "calculation",
    "type": "error"
  },
  {
    "query": "",
    "route": "error",
    "type": "error"
  }
]

Metrics:
{
  "total_queries": 8,
  "completed": 5,
  "completion_rate": "62%",
  "tool_calls_cost": 6
}


## Interactive mode

A `while True` REPL as the brief asks. It's wrapped in a function and called behind a
comment so the notebook can still be run top-to-bottom without `input()` blocking it —
uncomment the last line to chat with the agent, and type `exit` to stop.

In [9]:
def interactive():
    print("Agent ready. Type 'exit' to quit.")
    while True:
        try:
            user_input = input("You: ")
        except (EOFError, KeyboardInterrupt):
            print()
            break
        if user_input.strip().lower() == "exit":
            print("Bye!")
            break
        print("Agent:", json.dumps(agent(user_input)))


# interactive()   # <- uncomment to run the interactive loop

## Notes & possible improvements

**Working well**
- Clean routing for the three required intents plus a bonus tool, all returning the same
  JSON schema.
- Safe calculator (no raw `eval`), stop-word-aware keyword extraction.
- Retry wrapper, trajectory log and basic metrics, so the agent's behaviour is
  inspectable rather than a black box.

**Limitations / what I'd do next**
- Routing is keyword-based, so overlapping substrings need care (I used a word-boundary
  check for `count` so "account" doesn't misfire). A production system would use an LLM or
  a small classifier for intent — that's the single-agent-simulating-multi-agent idea from
  Q5.
- `parse_expression` only understands digits and basic operators; word problems like
  "twenty plus five" aren't handled.
- `METRICS` is in-memory; in a real deployment these would be pushed to a logger/DB to
  track cost over time.

Overall this maps straight onto the theory answers: the router is a stateful
directed-graph node, each tool is a node reached by a conditional edge, and the retry
wrapper is the cycle.